# Gikomba Fresh-Produce Stall — Sales & Inventory Analysis

**Analyst:** Owen Sichere  
**Cohort:** 1 Million Devs Cohort  
**Repository:** `plp-da-a1-gikomba-stall-analysis`  
**Tools:** Core Python only (no `pandas` or `numpy`)

This notebook analyzes 14 days of daily revenue (KES) and current stock records for Wanjiku's fresh-produce stall at Gikomba Market to evaluate overall performance, week-on-week growth, weekday vs. weekend demand, reorder urgency, and product gross margins.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]  # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Overall 14-Day Sales & Best/Worst Days

**Approach:** Sum the 14-day `daily_sales` list with `sum()` and divide by `len(daily_sales)` to find the daily average. Use `max()`, `min()`, and `list.index()` to locate the positions of the highest and lowest days, then map each index to its weekday name in `days` and compute the week number using integer floor division (`(index // 7) + 1`).

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)

highest_sale = max(daily_sales)
lowest_sale = min(daily_sales)

best_index = daily_sales.index(highest_sale)
worst_index = daily_sales.index(lowest_sale)

best_day = days[best_index]
best_week = (best_index // 7) + 1

worst_day = days[worst_index]
worst_week = (worst_index // 7) + 1

print(f"Total 14-day sales: KES {total_sales:,}")
print(f"Average daily sales: KES {average_sales:,.2f}")
print(f"Highest daily sales: KES {highest_sale:,} ({best_day}, Week {best_week})")
print(f"Lowest daily sales: KES {lowest_sale:,} ({worst_day}, Week {worst_week})")

Total 14-day sales: KES 46,450
Average daily sales: KES 3,317.86
Highest daily sales: KES 5,600 (Sat, Week 2)
Lowest daily sales: KES 1,950 (Tue, Week 1)


## Task 2: Week-on-Week Totals & Growth Percentage

**Approach:** Define `week_totals(sales)` using Python list slicing (`sales[:7]` for Week 1 and `sales[7:]` for Week 2) to return a `(week_1_total, week_2_total)` tuple. Unpack the returned tuple and calculate percentage growth relative to Week 1.

In [3]:
def week_totals(sales):
    """Return a tuple of (week_1_total, week_2_total) using list slicing."""
    week_1 = sum(sales[:7])
    week_2 = sum(sales[7:])
    return week_1, week_2


week1_total, week2_total = week_totals(daily_sales)
wow_growth = ((week2_total - week1_total) / week1_total) * 100

print(f"Week 1 Total: KES {week1_total:,}")
print(f"Week 2 Total: KES {week2_total:,}")
print(f"Week-on-Week Growth: +{wow_growth:.2f}% (KES {week2_total - week1_total:,} increase)")

Week 1 Total: KES 22,050
Week 2 Total: KES 24,400
Week-on-Week Growth: +10.66% (KES 2,350 increase)


## Task 3: Daily Performance Classification

**Approach:** Implement `classify_day(amount, average)` to label any day more than 20% above the 14-day average as `"Strong"`, more than 20% below average as `"Weak"`, and all others as `"Normal"`. Loop through all 14 days and count each category using the dictionary `.get()` counting pattern.

In [4]:
def classify_day(amount, average):
    """Classify a day's sales as 'Strong' (>20% above avg), 'Weak' (>20% below avg), or 'Normal'."""
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    return "Normal"


category_counts = {}
for day_name, amount in zip(days, daily_sales):
    category = classify_day(amount, average_sales)
    category_counts[category] = category_counts.get(category, 0) + 1

print("Day performance counts:", category_counts)

Day performance counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekday vs. Weekend Sales Comparison

**Approach:** Pair `days` and `daily_sales` with `zip()` to split observations into weekdays (`Mon`–`Fri`) and weekends (`Sat`–`Sun`). Calculate the mean revenue for each group and measure the weekend percentage uplift over the weekday average.

In [5]:
weekday_sales = [amount for day_name, amount in zip(days, daily_sales) if day_name not in ("Sat", "Sun")]
weekend_sales = [amount for day_name, amount in zip(days, daily_sales) if day_name in ("Sat", "Sun")]

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_uplift_pct = ((weekend_avg - weekday_avg) / weekday_avg) * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekend vs. Weekday difference: +{weekend_uplift_pct:.2f}%")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekend vs. Weekday difference: +82.50%


## Task 5: Stock Reorder Check & Gross Margin Analysis

**Approach:** Define `products_to_reorder(inventory)` to iterate through `stock.items()` and collect every item whose current `qty` is at or below its `reorder` threshold. In a second pass over `.items()`, compute each product's gross margin percentage using `(price - cost) / price * 100`.

In [6]:
def products_to_reorder(inventory):
    """Return a list of product names whose stock quantity is at or below the reorder level."""
    reorder_list = []
    for product, details in inventory.items():
        if details["qty"] <= details["reorder"]:
            reorder_list.append(product)
    return reorder_list


reorder_needed = products_to_reorder(stock)
gross_margins = {}
for product, details in stock.items():
    margin_pct = ((details["price"] - details["cost"]) / details["price"]) * 100
    gross_margins[product] = margin_pct

print("Products to reorder:", reorder_needed)
for product, margin in gross_margins.items():
    print(f"{product:<10} | Stock: {stock[product]['qty']:>2} kg | Reorder: {stock[product]['reorder']:>2} kg | Gross Margin: {margin:.2f}%")

Products to reorder: ['Tomatoes', 'Sukuma']
Tomatoes   | Stock: 18 kg | Reorder: 25 kg | Gross Margin: 33.33%
Onions     | Stock: 40 kg | Reorder: 20 kg | Gross Margin: 30.00%
Sukuma     | Stock:  9 kg | Reorder: 15 kg | Gross Margin: 40.00%
Potatoes   | Stock: 55 kg | Reorder: 30 kg | Gross Margin: 33.33%
Avocado    | Stock: 12 kg | Reorder: 10 kg | Gross Margin: 42.86%


## Task 6: Stall Performance Report

**Approach:** Combine all metrics from Tasks 1–5 into an aligned, executive-ready text summary using Python f-strings with explicit column widths and `:,` thousands separators on every KES value.

In [7]:
report_lines = [
    "=" * 62,
    f"{'WANJIKU FRESH PRODUCE — STALL PERFORMANCE REPORT':^62}",
    f"{'Gikomba Market | 14-Day Sales & Inventory Review':^62}",
    "=" * 62,
    f"{'Total 14-Day Revenue:':<32} KES {total_sales:>12,}",
    f"{'Daily Average Sales:':<32} KES {average_sales:>12,.2f}",
    f"{'Best Performing Day:':<32} KES {highest_sale:>12,} ({best_day}, Week {best_week})",
    f"{'Lowest Performing Day:':<32} KES {lowest_sale:>12,} ({worst_day}, Week {worst_week})",
    "-" * 62,
    "WEEKLY GROWTH & DAY SPLIT",
    f"{'Week 1 Total (Days 1-7):':<32} KES {week1_total:>12,}",
    f"{'Week 2 Total (Days 8-14):':<32} KES {week2_total:>12,}",
    f"{'Week-on-Week Growth:':<32} {wow_growth:>+15.2f}%",
    f"{'Weekday Average (Mon-Fri):':<32} KES {weekday_avg:>12,.2f}",
    f"{'Weekend Average (Sat-Sun):':<32} KES {weekend_avg:>12,.2f}",
    f"{'Weekend Sales Uplift:':<32} {weekend_uplift_pct:>+15.2f}%",
    "-" * 62,
    "DAILY PERFORMANCE CLASSIFICATION",
    f"  Strong (>20% above avg): {category_counts.get('Strong', 0)} days",
    f"  Normal (within +/-20%):  {category_counts.get('Normal', 0)} days",
    f"  Weak   (>20% below avg): {category_counts.get('Weak', 0)} days",
    "-" * 62,
    "INVENTORY & GROSS MARGIN SUMMARY",
    f"{'Product':<12} {'Stock':>7} {'Reorder':>9} {'Margin':>9}  {'Status'}",
]

for product, details in stock.items():
    status = "REORDER NOW" if product in reorder_needed else "OK"
    report_lines.append(
        f"{product:<12} {details['qty']:>4} kg {details['reorder']:>6} kg {gross_margins[product]:>8.2f}%  {status}"
    )

report_lines.extend([
    "-" * 62,
    f"Urgent Restock List: {', '.join(reorder_needed)}",
    "=" * 62,
])

print("\n".join(report_lines))

       WANJIKU FRESH PRODUCE — STALL PERFORMANCE REPORT       
       Gikomba Market | 14-Day Sales & Inventory Review       
Total 14-Day Revenue:            KES       46,450
Daily Average Sales:             KES     3,317.86
Best Performing Day:             KES        5,600 (Sat, Week 2)
Lowest Performing Day:           KES        1,950 (Tue, Week 1)
--------------------------------------------------------------
WEEKLY GROWTH & DAY SPLIT
Week 1 Total (Days 1-7):         KES       22,050
Week 2 Total (Days 8-14):        KES       24,400
Week-on-Week Growth:                      +10.66%
Weekday Average (Mon-Fri):       KES     2,685.00
Weekend Average (Sat-Sun):       KES     4,900.00
Weekend Sales Uplift:                     +82.50%
--------------------------------------------------------------
DAILY PERFORMANCE CLASSIFICATION
  Strong (>20% above avg): 4 days
  Normal (within +/-20%):  4 days
  Weak   (>20% below avg): 6 days
-----------------------------------------------------------

## Recommendations for Wanjiku

1. **Restock Tomatoes and Sukuma before Friday's rush:** Tomatoes (18 kg vs. 25 kg reorder level) and Sukuma (9 kg vs. 15 kg reorder level) are already below safe stock thresholds, and Sukuma carries a strong 40.00% gross margin that you cannot afford to lose to stockouts.
2. **Prepare extra stock and staffing for Saturdays and Sundays:** Weekend daily revenue averages KES 4,900.00—an 82.50% uplift over the weekday average of KES 2,685.00—with Week 2 Saturday peaking at KES 5,600, so bulk-buying high-turnover produce on Friday afternoon maximizes weekend sales.
3. **Promote Avocados alongside staples to lift total stall margin:** Avocados deliver the highest gross margin at 42.86% (KES 60 profit per kg), yet only 12 kg remain in stock (just 2 kg above the 10 kg reorder point); bundling Avocados with Tomatoes and Onions can boost weekday baskets above the KES 3,317.86 daily average.
4. **Target mid-week promotions on Mondays, Tuesdays, and Thursdays:** Six of the 14 days fell into the Weak category (>20% below average, bottoming out at KES 1,950 on Week 1 Tuesday), so introducing Tuesday/Thursday "fresh-bundle" discounts on well-stocked Potatoes (55 kg) and Onions (40 kg) can smooth weekday cash flow while preserving your +10.66% week-on-week growth momentum.